# 02 · Ask the frozen prior, then check the video

**Does the prior erase real motion, and can a cheap flow check already
fix it?** This notebook caches the evidence shared by all gate fits.
Prior and optical-flow weights stay frozen.

Run each notebook in a fresh kernel, in order **00 → 04**. Notebook 05 is
an external visual stress test. These are offline restoration experiments:
the model may inspect the declared complete clip. They do not claim causal
forecasting or clinical diagnosis.

**The default is real data.** Export `MP_RUN_ROOT`, the AMASS and GAVD data
paths, and the model configuration before opening Jupyter. See the
[launch guide](../../slurm/motion-preservation/README.md).
For a CPU walkthrough of the mechanics, explicitly choose `MP_MODE=demo`
and a separate run directory. Demo outputs cannot establish a research result.

[Proposal](../../docs/studies/motion-preservation/protocol/proposal.md)
· [Notebook guide](README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

project_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(project_override).expanduser()] if project_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)  # Resolve manifest/config paths from the checkout in every kernel.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, Video, display
from gavd6_sjepa.research_directions.motion_preservation import workflow, plots

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
cfg = workflow.config_from_environment()
RUN_ROOT = Path(cfg.run_root)
print(f"Mode: {cfg.mode}; run directory: {RUN_ROOT}")
if cfg.mode == "demo":
    display(Markdown("**DEMO ONLY: generated fixtures and stand-in models. "
                     "These outputs are not evidence about AMASS, GAVD, or a pretrained prior.**"))

## 1. What the prior and flow branches contribute

The motion prior proposes a repaired trajectory. Optical flow estimates
how visible image locations move between adjacent frames. For projected
path `q` and flow field `u`, the transport residual is

`r[t] = q[t+1] - q[t] - u[t](q[t])`.

Small residual means the path agrees with nearby image movement. It does
not prove anatomical correctness. A sleeve or shoe can move differently
from a joint center. Use robust local samples, quality evidence and
occlusion flags. Pose and flow both come from the same images and are
not independent sensors.

In [ ]:
# A numerical explanation of transport residuals, not an experiment result.
displacement = np.array([[4., 0.], [4., 0.]])
local_flow = np.array([[3.8, 0.2], [-1., 0.]])
residual = np.linalg.norm(displacement - local_flow, axis=-1)
display(pd.DataFrame({"candidate_dx": displacement[:, 0],
                      "nearby_flow_dx": local_flow[:, 0],
                      "transport_residual_pixels": residual},
                     index=["agreement example", "disagreement example"]))

## 2. Cache predictions on the declared development cases

The cache records the actual backend and reference origin. An imported
MoMask or MDM output must correspond to the same case and coordinate
convention. A demo smoother is not a released pretrained model.

The representation-only round trip is a separate baseline. If conversion
itself removes the event, that loss cannot be attributed entirely to
the pretrained prior. Keep its frame indices and report its error.

For real flow, start with a published checkpoint or a clearly labelled
alternate backend. Renderer-derived flow is an oracle diagnostic, never
an estimated-flow research result. Crop transforms apply to both flow
endpoints; camera compensation must be consistent on both trajectories.

Read reference flow error together with its coverage columns. The
conservative visibility check excludes occluded and unresolved surface
points. Foreground coverage is the fraction of visible source-body
pixels scored; all-pixel coverage uses the whole source image. A low
error on a small visible subset does not establish reliable whole-body
flow. MoMask inputs must use its trained rate of 20 frames per second.

In [ ]:
started = perf_counter()
predictions = workflow.cache_predictions(cfg, roles=("train", "calibration", "development"))
print(f"Cached/loaded {len(predictions):,} rows in {perf_counter() - started:.1f} seconds.")
display(predictions.head(16))

In [ ]:
backend_columns = [c for c in predictions if any(word in c.lower()
                   for word in ("method", "backend", "prior", "flow", "reference", "origin"))]
if backend_columns:
    display(predictions[backend_columns].drop_duplicates().head(30))
else:
    print("Read the cached metadata and configuration to identify the executed backends.")

## 3. Compare the inexpensive explanations first

Required first-pass comparisons are raw motion, the frozen prior,
smoothing, robust filtering, confidence gating, local flow propagation,
and a calibrated simple flow gate. The learned adapter should add value
beyond these. A reference-informed mixture is a diagnostic of what a
privileged choice can achieve. It is not a certified upper bound after
kinematic projection.

The proposal also calls for two-tracker disagreement, MFTIQ and compatible
HTD-Refine/H-MoRe/robust-prior-update comparisons. Check the implementation
ledger in the guide: unavailable external results remain missing
comparisons, not silently replaced or counted as executed baselines.

In [ ]:
figure = plots.preview_pair(cfg, split="train")
display(figure)
plt.close(figure)

## 4. Measure unmodified-prior event erasure before training

Compare the raw input, representation-only round trip, unmodified prior
and inexpensive baselines on development people now. This can reveal
that conversion causes the loss, that the prior does not erase the
selected event, or that a simple rule already repairs it.

These are full-strength, unmatched operating points. The table and plot
are mechanism diagnostics, not the later calibration-locked primary
comparison. Inspect achieved noise removal as well as retention.

In [ ]:
baseline_report = workflow.baseline_report(cfg, split="development")
display(baseline_report["summary"])
figure = plots.plot_tradeoff(baseline_report)
figure.axes[0].set_title("Full-strength baselines: unmatched repair quality")
display(figure)
plt.close(figure)

## 5. Does the proposed gate have room to help?

A mixture can retain some of the raw-minus-prior residual, but it cannot
invent a correct movement absent from both candidates. The
reference-informed mixture diagnostic uses known truth to choose each
weight before projection. Its post-projection score is not a mathematical
ceiling. Poor performance is a reason to inspect whether this gate design
has enough expressive power. If a simple flow rule succeeds equally well,
the full learned model needs a stronger reason to exist.

This is the 24-hour mechanism check. Do not tune using the final event
family. A held-person development result comes after calibration in
notebooks 03 and 04.

Next: [03 · Train and calibrate](03_train_and_calibrate.ipynb).